# 使用 `codex-cli` 实现 Jira ↔ GitHub 自动化

## 本 cookbook 的目的

本 cookbook 提供了一种实用的分步方法，用于实现 Jira 与 GitHub 之间工作流的自动化。只需给 Jira issue 打上标签，就能触发一个端到端流程：创建 **GitHub pull request**、保持两个系统同步更新，并简化代码审查，整个过程几乎无需手动操作。该自动化由运行在 GitHub Action 中的 [`codex-cli`](https://github.com/openai/openai-codex) 智能体驱动。

<img src="../../images/codex_action.png" alt="完整数据流图" width="500"/>


流程如下：

1. 给 Jira issue 打标签 
2. Jira Automation 调用 GitHub Action 
3. 该 Action 启动 `codex-cli` 来实现变更 
4. 创建一个 PR
5. Jira 状态流转并添加注释——形成一个简洁的零点击闭环。这包括更改工单状态、添加 PR 链接，以及在工单中评论更新信息。

## 前提条件

* Jira：项目管理员权限，以及创建自动化规则的能力  
* GitHub：写权限、添加仓库 secrets 的权限，以及一个受保护的 `main` 分支  
* 以仓库 secrets 形式配置的 API 密钥和 secrets：
  * `OPENAI_API_KEY` – 供 `codex-cli` 使用的 OpenAI 密钥  
  * `JIRA_BASE_URL`、`JIRA_EMAIL`、`JIRA_API_TOKEN` – 用于 Action 发起 REST 调用  
* 在本地安装 `codex-cli`（`pnpm add -g @openai/codex`），用于临时测试  
* 一个包含 `.github/workflows/` 文件夹的仓库


## 创建 Jira 自动化规则

<img src="../../images/jira_rule.png" alt="自动化规则" width="500"/>

该规则的第一步监听 issue 标签的变化。这样可以确保只有在添加或修改标签时才触发自动化——无需处理 issue 的每一次更新。

接下来，我们检查更新后的标签是否包含特定关键字，在本示例中使用的是 `aswe`。这起到过滤作用，只有明确标记为需要自动化的 issue 才会继续处理，从而避免无关更新带来的干扰。

如果条件满足，我们就向 GitHub 的 `workflow_dispatch` 端点发送一个 `POST` 请求。这会携带相关 issue 上下文启动一个 GitHub Actions 工作流。我们传入 issue key、摘要以及清理后的描述——对引号和换行符进行转义，以便负载能够在 YAML/JSON 中被正确解析。JIRA 中还有[更多字段](https://support.atlassian.com/cloud-automation/docs/jira-smart-values-issues/)可作为变量使用，以便在执行过程中为 codex 智能体提供更多上下文。

这种设置让团队能够严格控制哪些 Jira issue 会触发自动化，并确保 GitHub 收到结构化、干净的元数据来执行操作。我们还可以设置多个标签，每个标签触发不同的 GitHub Action。例如，一个标签可以启动快速修复 bug 的工作流，而另一个标签则可以开始重构代码或生成 API 桩代码。


## 添加 GitHub Action

GitHub Actions 允许你通过 YAML 文件定义工作流，在 GitHub 仓库中实现工作流自动化。这些工作流指定了一系列要执行的作业和步骤。无论是手动触发还是通过 POST 请求触发，GitHub 都会自动准备所需环境并运行定义好的工作流步骤。

为了处理来自 JIRA 的 `POST` 请求，我们将在仓库的 `.github/workflows/` 目录中创建一个 Github action，其 YAML 如下所示：


```yaml
name: Codex Automated PR
on:
  workflow_dispatch:
    inputs:
      issue_key:
        description: 'JIRA issue key (e.g., PROJ-123)'
        required: true
      issue_summary:
        description: 'Brief summary of the issue'
        required: true
      issue_description:
        description: 'Detailed issue description'
        required: true

permissions:
  contents: write           # allow the action to push code & open the PR
  pull-requests: write      # allow the action to create and update PRs

jobs:
  codex_auto_pr:
    runs-on: ubuntu-latest

    steps:
    # 0 – Checkout repository
    - uses: actions/checkout@v4
      with:
        fetch-depth: 0       # full history → lets Codex run tests / git blame if needed

    # 1 – Set up Node.js and Codex
    - uses: actions/setup-node@v4
      with:
        node-version: 22
    - run: pnpm add -g @openai/codex

    # 2 – Export / clean inputs (available via $GITHUB_ENV)
    - id: vars
      run: |
        echo "ISSUE_KEY=${{ github.event.inputs.issue_key }}"        >> $GITHUB_ENV
        echo "TITLE=${{ github.event.inputs.issue_summary }}"        >> $GITHUB_ENV
        echo "RAW_DESC=${{ github.event.inputs.issue_description }}" >> $GITHUB_ENV
        DESC_CLEANED=$(echo "${{ github.event.inputs.issue_description }}" | tr '\n' ' ' | sed 's/"/'\''/g')
        echo "DESC=$DESC_CLEANED"                                    >> $GITHUB_ENV
        echo "BRANCH=codex/${{ github.event.inputs.issue_key }}"     >> $GITHUB_ENV

    # 3 – Transition Jira issue to "In Progress"
    - name: Jira – Transition to In Progress
      env:
        ISSUE_KEY:      ${{ env.ISSUE_KEY }}
        JIRA_BASE_URL:  ${{ secrets.JIRA_BASE_URL }}
        JIRA_EMAIL:     ${{ secrets.JIRA_EMAIL }}
        JIRA_API_TOKEN: ${{ secrets.JIRA_API_TOKEN }}
      run: |
        curl -sS -X POST \
          --url   "$JIRA_BASE_URL/rest/api/3/issue/$ISSUE_KEY/transitions" \
          --user  "$JIRA_EMAIL:$JIRA_API_TOKEN" \
          --header 'Content-Type: application/json' \
          --data  '{"transition":{"id":"21"}}'
          # 21 is the transition ID for changing the ticket status to In Progress. Learn more here: https://developer.atlassian.com/cloud/jira/platform/rest/v3/api-group-issues/#api-rest-api-3-issue-issueidorkey-transitions-get

    # 4 – Set Git author for CI commits
    - run: |
        git config user.email "github-actions[bot]@users.noreply.github.com"
        git config user.name  "github-actions[bot]"

    # 5 – Let Codex implement & commit (no push yet)
    - name: Codex implement & commit
      env:
        CODEX_API_KEY:  ${{ secrets.OPENAI_API_KEY }}
      run: |
        set -e
        codex exec --sandbox workspace-write \
              "Implement JIRA ticket $ISSUE_KEY: $TITLE. $DESC"

        git add -A
        git commit -m "feat($ISSUE_KEY): $TITLE"

    # 6 – Open (and push) the PR in one go
    - id: cpr
      uses: peter-evans/create-pull-request@v6
      with:
        token:  ${{ secrets.GITHUB_TOKEN }}
        base:   main
        branch: ${{ env.BRANCH }}
        title:  "${{ env.TITLE }} (${{ env.ISSUE_KEY }})"
        body: |
          Auto-generated by Codex for JIRA **${{ env.ISSUE_KEY }}**.
          ---
          ${{ env.DESC }}

    # 7 – Transition Jira to "In Review" & drop the PR link
    - name: Jira – Transition to In Review & Comment PR link
      env:
        ISSUE_KEY:      ${{ env.ISSUE_KEY }}
        JIRA_BASE_URL:  ${{ secrets.JIRA_BASE_URL }}
        JIRA_EMAIL:     ${{ secrets.JIRA_EMAIL }}
        JIRA_API_TOKEN: ${{ secrets.JIRA_API_TOKEN }}
        PR_URL:         ${{ steps.cpr.outputs.pull-request-url }}
      run: |
        # Status transition
        curl -sS -X POST \
          --url   "$JIRA_BASE_URL/rest/api/3/issue/$ISSUE_KEY/transitions" \
          --user  "$JIRA_EMAIL:$JIRA_API_TOKEN" \
          --header 'Content-Type: application/json' \
          --data  '{"transition":{"id":"31"}}'
          # 31 is the Transition ID for changing the ticket status to In Review. Learn more here: https://developer.atlassian.com/cloud/jira/platform/rest/v3/api-group-issues/#api-rest-api-3-issue-issueidorkey-transitions-get

        # Comment with PR link
        curl -sS -X POST \
          --url   "$JIRA_BASE_URL/rest/api/3/issue/$ISSUE_KEY/comment" \
          --user  "$JIRA_EMAIL:$JIRA_API_TOKEN" \
          --header 'Content-Type: application/json' \
          --data  "{\"body\":{\"type\":\"doc\",\"version\":1,\"content\":[{\"type\":\"paragraph\",\"content\":[{\"type\":\"text\",\"text\":\"PR created: $PR_URL\"}]}]}}"
```


## 工作流中的关键步骤

1. **Codex 实现与提交**（第 5 步）
   - 使用 OpenAI API 实现 JIRA 工单的需求
   - 以非交互方式运行 Codex CLI，并授予工作区写权限
   - 使用标准化的提交信息提交所有更改

2. **创建 Pull Request**（第 6 步） 
   - 使用 peter-evans/create-pull-request action
   - 针对 main 分支创建 PR
   - 根据 JIRA 工单信息设置 PR 标题和描述
   - 返回 PR URL 供后续使用

3. **JIRA 更新**（第 7 步）
   - 通过 JIRA API 将工单流转到 "In Review" 状态
   - 在 JIRA 工单上发布包含 PR URL 的评论
   - 使用 curl 命令与 JIRA REST API 交互


## 给 Issue 打标签

为任意 bug/功能工单添加专用的 `aswe` 标签：

1. **创建时** – 在点击 *Create* 之前，在 "Labels" 字段中添加  
2. **已有 issue** – 将鼠标悬停在标签区域 → 点击铅笔图标 → 输入 `aswe`

<img src="../../images/add_label.png" alt="添加标签" width="500"/>


## 端到端流程

1. 添加 Jira 标签 → 触发 Automation
2. `workflow_dispatch` 被触发；Action 在 GitHub 上启动
3. `codex-cli` 编辑代码库并提交
4. 在生成的分支上创建 PR
5. Jira 被流转到 **In Review**，并发布一条包含 PR URL 的评论
6. 根据你常规的分支保护设置通知审查者

<img src="../../images/jira_comment.png" alt="包含 PR 链接的 Jira 评论" width="300"/>
<img src="../../images/jira_status_change.png" alt="Jira 状态流转到 In Review" width="300"/>


## 审查并合并 PR

你可以打开 JIRA 工单中发布的 PR 链接，检查一切是否正常，然后将其合并。如果你启用了分支保护和 Smart Commits 集成，那么在 pull request 合并后，Jira 工单会被自动关闭。

## 总结

该自动化在 Jira 与 GitHub 之间建立了无缝集成，从而简化你的开发工作流：

* **自动状态跟踪** - 工单在工作流中自动推进，无需手动更新
* **更好的开发者体验** - 专注于审查代码质量，而不是编写样板代码
* **减少交接摩擦** - 工单一打上标签，PR 就已准备好接受审查

`codex-cli` 工具是一款强大的 AI 编程助手，可以自动化重复性的编程任务。你可以在[这里](https://github.com/openai/codex/)了解更多相关信息